In [ ]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# recording
from lerobot.record import RecordConfig, DatasetRecordConfig, record
from lerobot.utils.utils import init_logging

from huggingface_hub import HfApi

# robot configs
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# utils
from dotenv import load_dotenv
from pathlib import Path

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

Recording Parameters:

In [2]:
NUM_EPISODES = 1
FPS = 30
EPISODE_TIME_SEC = 60 # how long each episode
RESET_TIME_SEC   = 10 # env reset time
TASK_DESCRIPTION = "pick the peg and place on the table"
REPO_NAME = 'test'
PUSH_TO_HUB = False

Login to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Robot Config:

In [4]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=2, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)
robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Dataset Config:

In [5]:
dataset_path = DATASETS_DIR / REPO_NAME
repo_id=f"{HF_NAME}/{REPO_NAME}"
root=f"{DATASETS_DIR}\\{REPO_NAME}"

resume = True if dataset_path.exists() else False

In [6]:
dscfg = DatasetRecordConfig(
    repo_id        = repo_id,
    single_task    = TASK_DESCRIPTION,
    root           = root,
    fps            = FPS,
    episode_time_s = EPISODE_TIME_SEC,
    reset_time_s   = RESET_TIME_SEC,
    push_to_hub    = PUSH_TO_HUB,
    video          = True,
    num_episodes   = NUM_EPISODES,
    num_image_writer_processes = 0,
    num_image_writer_threads_per_camera = 4
)

In [7]:
rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = teleop_config,
    policy       = None,
    display_data = True,
    resume       = resume
)

In [8]:
# init_logging(log_file=Path(DATASETS_DIR / 'logs' /'log.txt'))
record(rc)

WARNING 2025-09-22 16:03:37 deo_utils.py:36 'torchcodec' is not available in your platform, falling back to 'pyav' as a default decoder
Generating train split: 7726 examples [00:00, 252241.34 examples/s]
INFO 2025-09-22 16:03:43 a_opencv.py:178 OpenCVCamera(2) connected.
INFO 2025-09-22 16:04:57 a_opencv.py:178 OpenCVCamera(1) connected.
INFO 2025-09-22 16:04:57 follower.py:101 so_101_follower SO101Follower connected.
INFO 2025-09-22 16:04:57 01_leader.py:79 so_101_leader SO101Leader connected.
INFO 2025-09-22 16:04:57 ls\utils.py:227 Recording episode 10
WARNING 2025-09-22 16:04:58 t\record.py:269 FPS drop! 24.65/30
WARNING 2025-09-22 16:04:59 t\record.py:269 FPS drop! 27.78/30
WARNING 2025-09-22 16:05:01 t\record.py:269 FPS drop! 27.81/30
WARNING 2025-09-22 16:05:01 t\record.py:269 FPS drop! 23.47/30
WARNING 2025-09-22 16:05:02 t\record.py:269 FPS drop! 24.91/30
WARNING 2025-09-22 16:05:02 t\record.py:269 FPS drop! 23.31/30
WARNING 2025-09-22 16:05:02 t\record.py:269 FPS drop! 21.92/

KeyboardInterrupt: 

In [ ]:
# robot.disconnect()
# teleop.disconnect()